In [2]:
import pandas as pd
import numpy as np

In [29]:
import pandas as pd
import numpy as np

# Cargar bases
df_empleo = pd.read_csv("empleo_prov_trimestre_tratada.csv")
df_eph    = pd.read_csv("eph_unida.csv.xls")

# Mapear aglomerado -> provincia
mapping_aglo_prov = {
    2:  "buenos_aires",                       # Gran La Plata
    3:  "buenos_aires",                       # Bahía Blanca - Cerri
    4:  "santa_fe",                           # Gran Rosario
    5:  "santa_fe",                           # Gran Santa Fe
    6:  "entre_rios",                         # Gran Paraná
    7:  "misiones",                           # Posadas
    8:  "chaco",                              # Gran Resistencia
    9:  "chubut",                             # Cdro. Rivadavia – Rada Tilly
    10: "mendoza",                            # Gran Mendoza
    12: "corrientes",
    13: "cordoba",
    14: "entre_rios",                         # Concordia
    15: "formosa",
    17: "neuquen",
    18: "santiago_del_estero",
    19: "jujuy",
    20: "santa_cruz",
    22: "catamarca",
    23: "salta",
    25: "la_rioja",
    26: "san_luis",
    27: "san_juan",
    29: "tucuman",
    30: "la_pampa",
    31: "tierra_del_fuego",
    32: "ciudad_autonoma_de_buenos_aires",
    33: "buenos_aires",                       # Partidos del GBA
    34: "buenos_aires",                       # Mar del Plata - Batán
    36: "cordoba",                            # Río Cuarto
    38: "buenos_aires",                       # San Nicolás – Villa Constitución
    91: "chubut",                             # Rawson – Trelew
    93: "rio_negro",                          # Viedma – Carmen de Patagones
}

df_eph["provincia_completa"] = df_eph["AGLOMERADO"].map(mapping_aglo_prov)
df_eph = df_eph[~df_eph["provincia_completa"].isna()].copy()

# Tiempo en EPH
df_eph["anio"] = df_eph["ANO4"]
df_eph["trimestre"] = df_eph["TRIMESTRE"]

# === Variables micro en EPH ===

# aglomerado
df_eph["aglomerado"] = df_eph["AGLOMERADO"]

# género
df_eph["gen"] = df_eph["CH04"]
df_eph["mujer"] = (df_eph["CH04"] == 2).astype(int)

# edad
df_eph["edad"] = df_eph["CH06"]

# estado civil
df_eph["e_civil"] = df_eph["CH07"]
df_eph["e_civil_casado"] = (df_eph["CH07"] == 2).astype(int)
df_eph["e_civil_soltero"] = (df_eph["CH07"] == 1).astype(int)

# horas ocupación principal
df_eph["ht_ocup"] = df_eph["PP3E_TOT"]

# nivel educativo
df_eph["nivel_ed"] = df_eph["NIVEL_ED"]
df_eph["secundario_o_mas"] = np.where(df_eph["NIVEL_ED"].between(4, 8), 1, 0)

# condición de actividad
df_eph["cond_act"] = df_eph["ESTADO"]
df_eph["ocupado"] = (df_eph["ESTADO"] == 1).astype(int)
df_eph["desocupado"] = (df_eph["ESTADO"] == 2).astype(int)
df_eph["activo"] = df_eph["ESTADO"].isin([1, 2]).astype(int)

# parentesco con jefe
df_eph["cond_parent"] = df_eph["CH03"]
df_eph["es_jefe"] = (df_eph["CH03"] == 1).astype(int)
df_eph["es_hijo"] = (df_eph["CH03"] == 3).astype(int)

# alfabetismo
df_eph["alfabet"] = (df_eph["CH08"] == 1).astype(int)

# categoría ocupacional
df_eph["cat_ocup"] = df_eph["CAT_OCUP"]

# ingreso per cápita familiar
df_eph["percap_fam"] = df_eph["IPCF"]

# tamaño de aglomerado (> 500k hab.)
df_eph["tama_aglomerado"] = (df_eph["MAS_500"] == "S").astype(int)

# (placeholders para pobreza, si después definís una línea de pobreza real)
df_eph["nivel_pobreza"] = np.nan
df_eph["cond_pobre"] = np.nan

# === Collapse EPH a provincia-anio-trimestre ===

group_cols = ["provincia_completa", "anio", "trimestre"]

def wmean(x, w):
    x = x.astype(float)
    return np.average(x, weights=w) if w.sum() > 0 else np.nan

def agg_func(g):
    res = {}
    res["edad_prom"] = wmean(g["edad"], g["PONDERA"])
    res["prop_mujer"] = wmean(g["mujer"], g["PONDERA"])
    res["ipcf_prom"] = wmean(g["percap_fam"], g["PONDERA"])
    res["tasa_actividad"] = wmean(g["activo"], g["PONDERA"])
    res["tasa_empleo"] = wmean(g["ocupado"], g["PONDERA"])
    res["tasa_desocupacion"] = wmean(g["desocupado"], g["PONDERA"])
    res["prop_secundario_o_mas"] = wmean(g["secundario_o_mas"], g["PONDERA"])
    res["prop_alfabetos"] = wmean(g["alfabet"], g["PONDERA"])
    res["prop_jefes"] = wmean(g["es_jefe"], g["PONDERA"])
    res["prop_hijos"] = wmean(g["es_hijo"], g["PONDERA"])
    if g["ocupado"].sum() > 0:
        res["horas_prom_ocup"] = wmean(
            g.loc[g["ocupado"] == 1, "ht_ocup"],
            g.loc[g["ocupado"] == 1, "PONDERA"]
        )
    else:
        res["horas_prom_ocup"] = np.nan
    res["prop_aglo_mas_500k"] = wmean(g["tama_aglomerado"], g["PONDERA"])
    res["n_personas"] = g["PONDERA"].sum()
    return pd.Series(res)

agg = (
    df_eph
    .groupby(group_cols)
    .apply(agg_func)
    .reset_index()
)

# === Preparar base de empleo / tratamiento ===

df_empleo = df_empleo.rename(columns={
    "empleo_prom_trimestre": "empleo_prom",
    "salario_prom_trimestre": "salario_prom"
})
df_empleo["ln_empleo_prom"] = np.log(df_empleo["empleo_prom"])

# === Merge final para modelo ===

df_modelo = df_empleo.merge(
    agg,
    on=["provincia_completa", "anio", "trimestre"],
    how="inner"
)

# df_modelo es la base final para logit y random forest


C:\Users\Fede\AppData\Local\Temp\ipykernel_22192\1812758686.py:6: DtypeWarning: Columns (102,105,134,136,137,143,145,146) have mixed types. Specify dtype option on import or set low_memory=False.
  df_eph    = pd.read_csv("eph_unida.csv.xls")


In [33]:
df_modelo.describe()

,anio,trimestre,empleo_prom,salario_prom,provincia_tratada,ln_empleo_prom,edad_prom,prop_mujer,ipcf_prom,tasa_actividad,tasa_empleo,tasa_desocupacion,prop_secundario_o_mas,prop_alfabetos,prop_jefes,prop_hijos,horas_prom_ocup,prop_aglo_mas_500k,n_personas
count,240.000000,240.000000,240.000000,240.000000,240.000000,240.000000,240.000000,240.000000,240.000000,240.000000,240.000000,240.000000,240.000000,240.000000,240.000000,240.000000,240.000000,240.000000,2.400000e+02
mean,2017.800000,2.300000,2326.158428,28027.810645,0.825000,7.318513,34.292304,0.519913,8056.345752,0.435382,0.407555,0.027826,0.527785,0.643976,0.320501,0.389257,36.303077,0.285957,1.157936e+06
std,0.749895,1.102299,3284.436294,11000.466911,0.380761,0.822274,1.965611,0.011216,3003.662474,0.040454,0.033158,0.013643,0.052690,0.082601,0.035612,0.026800,2.853792,0.446979,2.778285e+06
min,2017.000000,1.000000,381.003623,13758.079254,0.000000,5.942809,30.727380,0.481917,2539.790820,0.318195,0.309072,0.004010,0.429418,0.456331,0.260960,0.284318,27.663403,0.000000,8.150400e+04
25%,2017.000000,1.000000,826.917798,20315.285752,1.000000,6.717705,32.888721,0.513114,6027.308265,0.416818,0.392713,0.016124,0.491270,0.591856,0.291964,0.378935,34.508434,0.000000,2.227802e+05
50%,2018.000000,2.000000,1369.935606,25352.073982,1.000000,7.222518,34.062685,0.519020,7638.971126,0.438742,0.408058,0.026002,0.521556,0.630569,0.317877,0.393477,36.295764,0.000000,3.637775e+05
75%,2018.000000,3.000000,2417.730516,32804.907231,1.000000,7.790574,35.217973,0.526554,9296.082344,0.455354,0.422627,0.039067,0.548566,0.700687,0.341327,0.407273,38.073402,0.915021,6.907648e+05
max,2019.000000,4.000000,17674.902622,83381.178194,1.000000,9.779901,41.891593,0.547531,17467.486999,0.573675,0.520927,0.060284,0.724846,0.877933,0.434531,0.437628,45.431394,1.000000,1.427310e+07


In [34]:
df_eph["provincia_completa"].value_counts(dropna=False)


buenos_aires                       138342
cordoba                             38544
santa_fe                            36066
entre_rios                          31808
salta                               27408
chubut                              25420
tucuman                             24901
ciudad_autonoma_de_buenos_aires     22265
mendoza                             21582
san_juan                            18718
catamarca                           18314
jujuy                               17780
santiago_del_estero                 17267
la_rioja                            16884
san_luis                            15859
corrientes                          14819
formosa                             14553
misiones                            14197
chaco                               13695
rio_negro                           13155
tierra_del_fuego                    11723
neuquen                             11170
la_pampa                            10113
santa_cruz                        

In [32]:
df_modelo.columns

Index(['provincia_completa', 'anio', 'trimestre', 'empleo_prom',
       'salario_prom', 'provincia_tratada', 'ln_empleo_prom', 'edad_prom',
       'prop_mujer', 'ipcf_prom', 'tasa_actividad', 'tasa_empleo',
       'tasa_desocupacion', 'prop_secundario_o_mas', 'prop_alfabetos',
       'prop_jefes', 'prop_hijos', 'horas_prom_ocup', 'prop_aglo_mas_500k',
       'n_personas'],
      dtype='object')

In [35]:
# 1) Ver NAs por columna (todas las columnas de df_modelo)
df_modelo.isna().sum()


provincia_completa       0
anio                     0
trimestre                0
empleo_prom              0
salario_prom             0
provincia_tratada        0
ln_empleo_prom           0
edad_prom                0
prop_mujer               0
ipcf_prom                0
tasa_actividad           0
tasa_empleo              0
tasa_desocupacion        0
prop_secundario_o_mas    0
prop_alfabetos           0
prop_jefes               0
prop_hijos               0
horas_prom_ocup          0
prop_aglo_mas_500k       0
n_personas               0
dtype: int64

# Modelos

In [36]:

# -----------------------------------
# 1. Elegir predictores 
# -----------------------------------

# Lista de variables explicativas que vamos a usar
cols_predictoras = [
    "empleo_prom",
    "salario_prom",
    "ln_empleo_prom",
    "edad_prom",
    "prop_mujer",
    "ipcf_prom",
    "tasa_actividad",
    "tasa_empleo",
    "tasa_desocupacion",
    "prop_secundario_o_mas",
    "prop_alfabetos",
    "prop_jefes",
    "prop_hijos",
    "horas_prom_ocup",
    "prop_aglo_mas_500k",
    "n_personas",
]

# Nos quedamos solo con las filas donde no haya NAs en Y ni en las X
df_modelo_ml = df_modelo.dropna(subset=cols_predictoras + ["provincia_tratada"]).copy()

X = df_modelo_ml[cols_predictoras]
y = df_modelo_ml["provincia_tratada"].astype(int)

print(X.shape, y.shape)


(240, 16) (240,)


In [37]:
import statsmodels.api as sm

# Agregamos constante
X_const = sm.add_constant(X)

logit_model = sm.Logit(y, X_const)
logit_res = logit_model.fit()

print(logit_res.summary())


Optimization terminated successfully.
         Current function value: 0.243591
         Iterations 11
                           Logit Regression Results                           
Dep. Variable:      provincia_tratada   No. Observations:                  240
Model:                          Logit   Df Residuals:                      224
Method:                           MLE   Df Model:                           15
Date:                Sat, 06 Dec 2025   Pseudo R-squ.:                  0.4747
Time:                        12:35:40   Log-Likelihood:                -58.462
converged:                       True   LL-Null:                       -111.29
Covariance Type:            nonrobust   LLR p-value:                 1.090e-15
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
const                   -48.3987     24.148     -2.004      0.045     -95.728      -1

In [38]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

# Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=123,
    stratify=y
)

# Modelo Random Forest
rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=123,
    class_weight="balanced"  # por si hay desbalance en provincia_tratada
)

rf.fit(X_train, y_train)

# Métricas básicas
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, y_prob))

# Importancia de variables
importances = pd.Series(rf.feature_importances_, index=cols_predictoras).sort_values(ascending=False)
print(importances)


              precision    recall  f1-score   support

           0       0.89      0.62      0.73        13
           1       0.92      0.98      0.95        59

    accuracy                           0.92        72
   macro avg       0.90      0.80      0.84        72
weighted avg       0.91      0.92      0.91        72

AUC: 0.9895697522816167
prop_secundario_o_mas    0.115473
salario_prom             0.111017
prop_alfabetos           0.091591
empleo_prom              0.080954
ln_empleo_prom           0.076807
ipcf_prom                0.074359
tasa_actividad           0.068296
horas_prom_ocup          0.066063
tasa_empleo              0.059823
prop_jefes               0.056232
prop_hijos               0.049161
n_personas               0.044129
edad_prom                0.043640
prop_mujer               0.031843
tasa_desocupacion        0.027337
prop_aglo_mas_500k       0.003275
dtype: float64
